# 01_build_cohort (NACC) — Updated

Builds a participant-level cohort from `data/investigator_nacc72.csv` and assigns 5-year outcome labels using dictionary semantics:
- AD: `NACCALZP`
- Vascular (UDS v3.x): `CVD`, `CVDIF`
- PD/LBD: `NACCLBDP`, `NACCLBDE`

Outputs: `output/cohort.csv`


In [16]:
import os
import re
import numpy as np
import pandas as pd

DATA_DIR = os.path.join("..", "data")
OUT_DIR  = os.path.join("..", "output")
os.makedirs(OUT_DIR, exist_ok=True)

NACC_MAIN_PATH = os.path.join(DATA_DIR, "investigator_nacc72.csv")
NACC_CSF_PATH  = os.path.join(DATA_DIR, "investigator_fcsf_nacc72.csv")
DICT_PATH      = os.path.join(DATA_DIR, "nacc_variable_dictionary_consolidated.xlsx")

HORIZON_YEARS = 5

print("Main:", NACC_MAIN_PATH)
print("CSF :", NACC_CSF_PATH)
print("Dict:", DICT_PATH)


Main: ..\data\investigator_nacc72.csv
CSF : ..\data\investigator_fcsf_nacc72.csv
Dict: ..\data\nacc_variable_dictionary_consolidated.xlsx


## Dictionary (optional) + cleaning helpers

In [17]:
import os
import re
import numpy as np
import pandas as pd

STANDARD_MISSING = {-4, 9}


def clean_coded(series: pd.Series, extra_missing=None):
    s = pd.to_numeric(series, errors="coerce")
    miss = set(STANDARD_MISSING)
    if extra_missing:
        miss |= set(extra_missing)
    return s.mask(s.isin(list(miss)), np.nan)


def _normalize_allowable_text(s: str) -> str:
    return s.replace(" - ", " -").replace("- ", "-")

def parse_code_labels(allowable_codes: str):
    """Parse strings like '0 = ... 1 = ... 9 = Missing -4 = Not available' into [(code,label), ...]."""
    if not isinstance(allowable_codes, str) or not allowable_codes.strip():
        return []
    txt = _normalize_allowable_text(allowable_codes)
    parts = re.split(r'(?=(?:^|\s)(-?\d+)\s*=\s*)', txt)

    out = []
    if len(parts) < 3:
        return out

    i = 1
    while i < len(parts) - 1:
        try:
            code = int(parts[i])
        except Exception:
            i += 2
            continue
        rest = parts[i + 1]
        label = re.sub(r'\s+', ' ', str(rest)).strip().strip(" -;")
        out.append((code, label))
        i += 2
    return out

def _pick_columns(df: pd.DataFrame):
    """Find best guess for variable-name column and allowable-codes column."""
    cols = {c.lower().strip(): c for c in df.columns}

    var_candidates = [
        "variable", "varname", "name", "field", "field_name", "variable_name"
    ]
    allow_candidates = [
        "allowable_codes", "allowable code", "allowed_values", "allowed values",
        "codes", "values", "allowablecodes", "allowedvalues"
    ]

    var_col = next((cols[c] for c in var_candidates if c in cols), None)
    allow_col = next((cols[c] for c in allow_candidates if c in cols), None)
    return var_col, allow_col

def build_missing_code_map_debug(dict_xlsx_path: str):
    """
    Debug-friendly version:
    - Raises clear errors if file/sheets/columns aren't found.
    - Searches across all sheets for columns if needed.
    """
    if not os.path.exists(dict_xlsx_path):
        raise FileNotFoundError(
            f"Dictionary file not found at: {dict_xlsx_path}\n"
            f"Make sure it is placed in ./data/ and the name matches exactly."
        )

    xl = pd.ExcelFile(dict_xlsx_path)
    print("✅ Dictionary file opened.")
    print("Sheets:", xl.sheet_names)

    # 1) Try preferred sheet names first
    preferred = {"dictionary", "data dictionary", "dict"}
    sheet_order = []
    for s in xl.sheet_names:
        if s.lower().strip() in preferred:
            sheet_order.insert(0, s)
        else:
            sheet_order.append(s)

    chosen_sheet = None
    chosen_var_col = None
    chosen_allow_col = None
    chosen_df = None

    # 2) Find first sheet that contains suitable columns
    for s in sheet_order:
        tmp = pd.read_excel(dict_xlsx_path, sheet_name=s)
        var_col, allow_col = _pick_columns(tmp)
        if var_col is not None and allow_col is not None:
            chosen_sheet = s
            chosen_var_col = var_col
            chosen_allow_col = allow_col
            chosen_df = tmp
            break

    if chosen_sheet is None:
        # print columns for debugging
        for s in xl.sheet_names:
            tmp = pd.read_excel(dict_xlsx_path, sheet_name=s)
            print(f"\nSheet '{s}' columns:\n{list(tmp.columns)}")
        raise ValueError(
            "Could not find required columns in ANY sheet.\n"
            "Need columns like 'variable' and 'allowable_codes' (names may differ). "
            "See printed columns above."
        )

    print(f"\n✅ Using sheet: '{chosen_sheet}'")
    print(f"   var column:   {chosen_var_col}")
    print(f"   codes column: {chosen_allow_col}")

    d = chosen_df

    missing_map = {}
    label_map = {}

    for _, r in d.iterrows():
        var = r.get(chosen_var_col, None)
        allow = r.get(chosen_allow_col, "")

        if not isinstance(var, str) or not var.strip():
            continue

        pairs = parse_code_labels(allow)
        code_to_label = {c: lbl for c, lbl in pairs}
        label_map[var] = code_to_label

        miss = set()
        for c, lbl in pairs:
            l = str(lbl).lower()
            if ("missing" in l) or ("not available" in l) or ("not applicable" in l) or ("unknown" in l):
                miss.add(c)
        missing_map[var] = miss

    meta = (chosen_sheet, chosen_var_col, chosen_allow_col, len(label_map))
    return missing_map, label_map, meta

# --- Run debug load ---
missing_map, label_map, meta = build_missing_code_map_debug(DICT_PATH)
print("\nDictionary read OK:", meta)

# Quick sanity checks
for v in ["COGSTAT", "CVDIF", "NACCALZP", "NACCLBDP", "NACCUDSD"]:
    ex = list(label_map.get(v, {}).items())[:8]
    print(f"{v} example codes:", ex if ex else "(not found in dictionary)")


✅ Dictionary file opened.
Sheets: ['Dictionary', 'RawSections']

✅ Using sheet: 'Dictionary'
   var column:   variable
   codes column: allowable_codes

Dictionary read OK: ('Dictionary', 'variable', 'allowable_codes', 1044)
COGSTAT example codes: [(0, '0 = Clinician unable to render opinion Not applic Not applic (Form C1 or C2)'), (1, '1 = Better than normal for age TI question no. 13a (Form C2T)'), (2, '2 = Normal for age'), (3, '3 = One or two test scores abnormal TF question no. Not applic Not applic 13a (Form C2T)'), (4, '4 = Three or more scores are abnormal or lower than expected'), (9, '9 = Missing'), (-4, '4 = Not available: UDS form submitted did not collect data in this way, or a skip pattern precludes response to this question')]
CVDIF example codes: [(1, '1 = Primary TI question no. 15a'), (2, '2 = Contributing TF question no. Not applic 15a'), (3, '3 = Non-contributing'), (7, '7 = Cognitively impaired but no VBI diagnosis'), (8, '8 = Diagnosis of normal cognition'), (-4, 

## Load main file + visit_date

In [18]:
df = pd.read_csv(NACC_MAIN_PATH, low_memory=False)
df["NACCID"] = df["NACCID"].astype(str)

def make_visit_date(df_: pd.DataFrame) -> pd.Series:
    y = pd.to_numeric(df_["VISITYR"], errors="coerce")
    m = pd.to_numeric(df_["VISITMO"], errors="coerce").fillna(6).astype(int).clip(1, 12)
    d = pd.to_numeric(df_["VISITDAY"], errors="coerce").fillna(15).astype(int).clip(1, 28)
    return pd.to_datetime(dict(year=y, month=m, day=d), errors="coerce")

df["visit_date"] = make_visit_date(df)
df = df.dropna(subset=["NACCID", "visit_date"]).copy()
df = df.sort_values(["NACCID", "visit_date"]).reset_index(drop=True)

if "FORMVER" in df.columns:
    df["FORMVER"] = pd.to_numeric(df["FORMVER"], errors="coerce")

print("Rows:", len(df), "Participants:", df["NACCID"].nunique())
df[["NACCID","visit_date","FORMVER"]].head()


Rows: 207454 Participants: 55268


,NACCID,visit_date,FORMVER
0,NACC000011,2006-04-17,1.0
1,NACC000011,2007-06-18,1.0
2,NACC000011,2008-06-03,2.0
3,NACC000011,2009-08-03,2.0
4,NACC000034,2015-07-16,3.0


In [19]:
print("NACCUDSD in columns?", "NACCUDSD" in df.columns)
print("NACCCUDSD in columns?", "NACCCUDSD" in df.columns)

# show close matches
[c for c in df.columns if "UDSD" in c]


NACCUDSD in columns? True
NACCCUDSD in columns? False


['NACCUDSD']

In [32]:
cand = ["BPSYS","BPDIAS","CVCHF","CVANGINA","CVHATT","PREVSTK","STROKCOG","CVDCOG","STKIMAG"]
print({c: (c in df.columns) for c in cand})

# optional: see similar column names if spelling differs
[key for key in df.columns if "BP" in key.upper()][:50]
[key for key in df.columns if "STROK" in key.upper()][:50]
[key for key in df.columns if "CVD" in key.upper()][:50]


{'BPSYS': True, 'BPDIAS': True, 'CVCHF': True, 'CVANGINA': True, 'CVHATT': True, 'PREVSTK': True, 'STROKCOG': True, 'CVDCOG': True, 'STKIMAG': True}


['CVDCOG',
 'CVDIMAG',
 'CVDIMAG1',
 'CVDIMAG2',
 'CVDIMAG3',
 'CVDIMAG4',
 'CVDIMAGX',
 'CVDSIGNS',
 'CVDMOTL',
 'CVDMOTR',
 'CVD',
 'CVDIF']

## Optional: restrict to UDS v3 / 3.2 only

In [20]:
USE_ONLY_V3 = False  # set True to keep only FORMVER in {3, 3.2}

if USE_ONLY_V3 and "FORMVER" in df.columns:
    before = len(df)
    df = df[df["FORMVER"].isin([3, 3.2])].copy()
    df = df.sort_values(["NACCID","visit_date"]).reset_index(drop=True)
    print("Filtered to FORMVER in {3,3.2}: rows", before, "->", len(df))
else:
    print("No FORMVER filtering applied.")


No FORMVER filtering applied.


## Clean key status + etiology variables

In [21]:
STATUS_VARS = ["NACCUDSD", "DEMENTED", "NORMCOG", "NACCTMCI", "COGSTAT"]
AD_VARS   = ["NACCALZD", "NACCALZP"]
VBI_VARS  = ["CVD", "CVDIF"]
LBD_VARS  = ["NACCLBDE", "NACCLBDP"]
OLD_VASC_VARS = ["VASC", "VASCIF", "VASCPS", "VASCPSIF", "STROKE", "STROKIF"]

for v in STATUS_VARS + AD_VARS + VBI_VARS + LBD_VARS + OLD_VASC_VARS:
    if v in df.columns:
        extra = missing_map.get(v, set()) if missing_map else set()
        df[v] = clean_coded(df[v], extra_missing=extra)

# COGSTAT: 0 = clinician unable to render opinion -> treat as missing
if "COGSTAT" in df.columns:
    df.loc[df["COGSTAT"] == 0, "COGSTAT"] = np.nan

df[[c for c in ["NACCUDSD","DEMENTED","NORMCOG","NACCTMCI","COGSTAT",
                "NACCALZD","NACCALZP","CVD","CVDIF","NACCLBDE","NACCLBDP"]
    if c in df.columns]].head()


,NACCUDSD,DEMENTED,NORMCOG,NACCTMCI,COGSTAT,NACCALZD,NACCALZP,CVD,CVDIF,NACCLBDE,NACCLBDP
0,3,0,0,3,3.0,0,7,NaN,NaN,0,7
1,3,0,0,3,3.0,0,7,NaN,NaN,0,7
2,3,0,0,1,3.0,1,1,NaN,NaN,0,7
3,2,0,0,8,2.0,0,7,NaN,NaN,0,7
4,3,0,0,1,2.0,1,1,0.0,7.0,0,7


## Baseline selection (CN or MCI; not demented)

In [22]:
def first_row_per_group(df_: pd.DataFrame, group_col: str, sort_col: str) -> pd.DataFrame:
    return df_.sort_values([group_col, sort_col]).groupby(group_col, as_index=False).first()

if "NACCUDSD" in df.columns:
    cnmci_mask = df["NACCUDSD"].isin([1, 3])
else:
    cn_mask  = (df["NORMCOG"] == 1) if "NORMCOG" in df.columns else False
    mci_mask = df["NACCTMCI"].isin([1,2,3,4]) if "NACCTMCI" in df.columns else False
    cnmci_mask = cn_mask | mci_mask

no_dem_mask = (df["DEMENTED"] == 0) if "DEMENTED" in df.columns else True

baseline_candidates = df[cnmci_mask & no_dem_mask].copy()
baseline = first_row_per_group(baseline_candidates, "NACCID", "visit_date").copy()
baseline = baseline.rename(columns={"visit_date": "baseline_date"})

print("Baseline participants:", len(baseline))
baseline[["NACCID","baseline_date"] + [c for c in ["NACCUDSD","DEMENTED"] if c in baseline.columns]].head()


C:\Users\branjan\AppData\Local\Temp\ipykernel_20816\2008667892.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return df_.sort_values([group_col, sort_col]).groupby(group_col, as_index=False).first()


Baseline participants: 37007


,NACCID,baseline_date,NACCUDSD,DEMENTED
0,NACC000011,2006-04-17,3,0
1,NACC000034,2015-07-16,3,0
2,NACC000067,2012-05-23,3,0
3,NACC000073,2016-05-20,1,0
4,NACC000144,2010-08-17,1,0


## 5-year window, censoring, first dementia visit

In [23]:
def years_between(a: pd.Series, b: pd.Series) -> pd.Series:
    return (b - a).dt.days / 365.25

keep_ids = set(baseline["NACCID"])
df2 = df[df["NACCID"].isin(keep_ids)].merge(
    baseline[["NACCID","baseline_date"]], on="NACCID", how="left"
).copy()

df2["t_years"] = years_between(df2["baseline_date"], df2["visit_date"])
post = df2[df2["t_years"] > 0].copy()

max_follow = post.groupby("NACCID")["t_years"].max().rename("max_follow_years")
cohort = baseline.merge(max_follow, on="NACCID", how="left")
cohort["max_follow_years"] = cohort["max_follow_years"].fillna(0)
cohort["has_5y_followup"] = cohort["max_follow_years"] >= HORIZON_YEARS

window = post[post["t_years"] <= HORIZON_YEARS].copy()

if "NACCUDSD" in window.columns:
    dem_events = window[window["NACCUDSD"] == 4].copy()
else:
    dem_events = window[window["DEMENTED"] == 1].copy()

first_dem = dem_events.sort_values(["NACCID","visit_date"]).groupby("NACCID", as_index=False).first()
first_dem = first_dem.rename(columns={"visit_date":"dementia_date"})

print("Participants with dementia within 5y:", first_dem["NACCID"].nunique())
first_dem[["NACCID","dementia_date"] + [c for c in ["FORMVER","NACCUDSD","DEMENTED"] if c in first_dem.columns]].head()


Participants with dementia within 5y: 3852


C:\Users\branjan\AppData\Local\Temp\ipykernel_20816\3080742716.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  first_dem = dem_events.sort_values(["NACCID","visit_date"]).groupby("NACCID", as_index=False).first()


,NACCID,dementia_date,FORMVER,NACCUDSD,DEMENTED
0,NACC000271,2009-05-14,2.0,4,1
1,NACC000333,2024-07-24,3.0,4,1
2,NACC000626,2024-06-10,3.0,4,1
3,NACC001456,2024-03-08,3.0,4,1
4,NACC002043,2012-03-28,2.0,4,1


## Etiology labeling (PRIMARY only) — updated from your dictionary screenshots

In [24]:
MISSING = {-4, 9}

def _num(x):
    x = pd.to_numeric(x, errors="coerce")
    if pd.isna(x) or x in MISSING:
        return np.nan
    return x

def _is1(x):
    x = _num(x)
    return (not np.isnan(x)) and (x == 1)

def _in_set(x, s):
    x = _num(x)
    return (not np.isnan(x)) and (x in s)

def _in_ok(x, ok):
    return _in_set(x, ok)

def etiology_one_label_priority_compromise(row: pd.Series) -> str:
    """
    Single-label outcome with priority: AD > PD/LBD > Vascular.
    Compromise definitions:
      - AD positive only if PRIMARY (NACCALZP == 1)
      - LBD positive only if PRIMARY (NACCLBDP == 1)
      - Vascular positive if PRIMARY OR CONTRIBUTING (IF in {1,2}) OR fallback evidence
    """
    # AD/LBD primary only
    has_ad  = (_num(row.get("NACCALZP")) == 1)
    has_lbd = (_num(row.get("NACCLBDP")) == 1)

    # Vascular primary or contributing
    ok_v = {1, 2}
    fv = _num(row.get("FORMVER"))
    has_vasc = False

    if (not np.isnan(fv)) and (fv >= 3):
        has_vasc = _is1(row.get("CVD")) and _in_set(row.get("CVDIF"), ok_v)
    else:
        has_vasc = _is1(row.get("VASC")) and (_in_set(row.get("VASCIF"), ok_v) or _in_set(row.get("VASCPSIF"), ok_v))

    # Fallback vascular evidence
    if not has_vasc:
        has_vasc = any([
            _is1(row.get("CVDCOG")),
            _is1(row.get("STROKCOG")),
            _is1(row.get("PREVSTK")),
            _is1(row.get("STROKE")),
            _is1(row.get("CBSTROKE")),
            _is1(row.get("CBTIA")),
            _is1(row.get("CVDIMAG")),
            _is1(row.get("STKIMAG")),
        ])

    # Priority tie-breaker
    if has_ad:
        return "AD"
    if has_lbd:
        return "PD_LBD"
    if has_vasc:
        return "Vascular"
    return "MixedOrOther"

first_dem["etiology_primary"] = first_dem.apply(etiology_one_label_priority_compromise, axis=1)
first_dem["etiology_primary"].value_counts(dropna=False)


etiology_primary
AD              2948
MixedOrOther     445
PD_LBD           283
Vascular         176
Name: count, dtype: int64

In [25]:
# --- ensure VASC primary flag exists (robust across UDS versions) ---
MISSING = {-4, 9}

def vasc_primary_flag(row):
    formver = _num(row.get("FORMVER"))
    if (not np.isnan(formver)) and (formver >= 3):
        # UDS v3.x vascular brain injury
        return _is1(row.get("CVD")) and _in_set(row.get("CVDIF"), {1})
    else:
        # UDS v1/v2 vascular dementia
        return _is1(row.get("VASC")) and (_in_set(row.get("VASCIF"), {1}) or _in_set(row.get("VASCPSIF"), {1}))

if "flag_VASC_primary" not in first_dem.columns:
    first_dem["flag_VASC_primary"] = first_dem.apply(vasc_primary_flag, axis=1)

# AD primary only
first_dem["flag_AD_primary"] = (pd.to_numeric(first_dem["NACCALZP"], errors="coerce") == 1)

# Make sure they're boolean
v = first_dem["flag_VASC_primary"].fillna(False).astype(bool)
a = first_dem["flag_AD_primary"].fillna(False).astype(bool)

print("VASC primary & AD primary:", (v & a).sum())


VASC primary & AD primary: 0


In [26]:
tmp = first_dem.copy()

tmp["vasc_primary_rule"] = tmp.apply(lambda r: (
    (_is1(r.get("CVD")) and _in_set(r.get("CVDIF"), {1})) 
    if ((not np.isnan(_num(r.get("FORMVER")))) and (_num(r.get("FORMVER")) >= 3))
    else (_is1(r.get("VASC")) and (_in_set(r.get("VASCIF"), {1}) or _in_set(r.get("VASCPSIF"), {1})))
), axis=1)

print("Vascular by PRIMARY rule:", int(tmp["vasc_primary_rule"].sum()))
print("Vascular final labeled:", int((tmp["etiology_primary"]=="Vascular").sum()))

Vascular by PRIMARY rule: 99
Vascular final labeled: 176


In [27]:
def has_ad_primary(r):
    return (_num(r.get("NACCALZP")) == 1)

def has_lbd_primary(r):
    return (_num(r.get("NACCLBDP")) == 1)

def has_vasc_primary_or_contrib(r):
    fv = _num(r.get("FORMVER"))
    if (not np.isnan(fv)) and (fv >= 3):
        return _is1(r.get("CVD")) and _in_set(r.get("CVDIF"), {1,2})
    return _is1(r.get("VASC")) and (_in_set(r.get("VASCIF"), {1,2}) or _in_set(r.get("VASCPSIF"), {1,2}))

def has_vasc_fallback(r):
    return any([
        _is1(r.get("CVDCOG")),
        _is1(r.get("STROKCOG")),
        _is1(r.get("PREVSTK")),
        _is1(r.get("STROKE")),
        _is1(r.get("CBSTROKE")),
        _is1(r.get("CBTIA")),
        _is1(r.get("CVDIMAG")),
        _is1(r.get("STKIMAG")),
    ])

first_dem["flag_AD_primary"]  = first_dem.apply(has_ad_primary, axis=1).fillna(False).astype(bool)
first_dem["flag_LBD_primary"] = first_dem.apply(has_lbd_primary, axis=1).fillna(False).astype(bool)
first_dem["flag_VASC_pc"]     = first_dem.apply(has_vasc_primary_or_contrib, axis=1).fillna(False).astype(bool)
first_dem["flag_VASC_any"]    = (first_dem["flag_VASC_pc"] | first_dem.apply(has_vasc_fallback, axis=1).fillna(False).astype(bool))

print("AD primary-positive:", first_dem["flag_AD_primary"].sum())
print("LBD primary-positive:", first_dem["flag_LBD_primary"].sum())
print("VASC (primary|contrib)-positive:", first_dem["flag_VASC_pc"].sum())
print("VASC any-positive:", first_dem["flag_VASC_any"].sum())

print("\nOverlap counts:")
print("VASC pc & AD primary:", (first_dem["flag_VASC_pc"] & first_dem["flag_AD_primary"]).sum())
print("VASC pc & LBD primary:", (first_dem["flag_VASC_pc"] & first_dem["flag_LBD_primary"]).sum())
print("VASC pc & (AD or LBD):", (first_dem["flag_VASC_pc"] & (first_dem["flag_AD_primary"] | first_dem["flag_LBD_primary"])).sum())


AD primary-positive: 2948
LBD primary-positive: 283
VASC (primary|contrib)-positive: 267
VASC any-positive: 553

Overlap counts:
VASC pc & AD primary: 141
VASC pc & LBD primary: 4
VASC pc & (AD or LBD): 145


## Final label assignment (4-class + Mixed/Censored helpers)

In [28]:
cohort = cohort.merge(
    first_dem[["NACCID","dementia_date","etiology_primary"]],
    on="NACCID", how="left"
)
cohort["has_dementia_5y"] = cohort["dementia_date"].notna()

def assign_label(r):
    if r["has_dementia_5y"]:
        if r["etiology_primary"] in {"AD","Vascular","PD_LBD"}:
            return r["etiology_primary"]
        return "MixedOrOther"
    if r["has_5y_followup"]:
        return "NoDementia_5y"
    return "Censored"

cohort["label"] = cohort.apply(assign_label, axis=1)
cohort["label"].value_counts(dropna=False)


label
Censored         22339
NoDementia_5y    10816
AD                2948
MixedOrOther       445
PD_LBD             283
Vascular           176
Name: count, dtype: int64

## Build baseline feature table (starter) + optional CSF merge

In [33]:
STARTER_FEATURES = [
    "NACCID","baseline_date",
    "NACCAGE","SEX","RACE","HISPANIC","EDUC",

    # (recommended: keep cognitive status vars OUT of predictors)
    # "NACCUDSD","COGSTAT","NORMCOG","NACCTMCI",

    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","APNEA","DEP2YRS",
    "NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON",
    "CDRSUM","CDRGLOB",
    # Vascular / cardio / BP features (available in your df)
    "BPSYS","BPDIAS",
    "CVCHF","CVANGINA","CVHATT",
    "PREVSTK","STROKCOG","CVDCOG","STKIMAG",
    # optional extra imaging evidence (since you showed they exist)
    "CVDIMAG","CVDIMAG1","CVDIMAG2","CVDIMAG3","CVDIMAG4",
]

cols = [c for c in STARTER_FEATURES if c in baseline.columns]
X = baseline[cols].copy()

X = X.merge(
    cohort[["NACCID","baseline_date","label","has_5y_followup","max_follow_years"]],
    on=["NACCID","baseline_date"], how="left"
)

NUM_VARS = ["NACCAGE","EDUC","NACCMMSE","NACCMOCA","TRAILA","TRAILB","CDRSUM","CDRGLOB"]
for v in NUM_VARS:
    if v in X.columns:
        extra = missing_map.get(v, set()) if missing_map else set()
        X[v] = clean_coded(X[v], extra_missing=extra)

for v in ["TRAILB","TRAILA","NACCMMSE","NACCMOCA","CDRSUM"]:
    if v in X.columns:
        X[f"{v}_missing"] = X[v].isna().astype(int)

VASC_EXTRA = [
    "BPSYS","BPDIAS",
    "CVCHF","CVANGINA","CVHATT",
    "PREVSTK","STROKCOG","CVDCOG","STKIMAG",
    "CVDIMAG","CVDIMAG1","CVDIMAG2","CVDIMAG3","CVDIMAG4"
]

for v in VASC_EXTRA:
    if v in X.columns:
        extra = missing_map.get(v, set()) if missing_map else set()
        X[v] = clean_coded(X[v], extra_missing=extra)

for v in ["BPSYS","BPDIAS","STROKCOG","CVDCOG","STKIMAG","CVDIMAG"]:
    if v in X.columns:
        X[f"{v}_missing"] = X[v].isna().astype(int)

display(X.head())


,NACCID,baseline_date,NACCAGE,SEX,RACE,HISPANIC,EDUC,DIABETES,HYPERTEN,HYPERCHO,...,TRAILA_missing,NACCMMSE_missing,NACCMOCA_missing,CDRSUM_missing,BPSYS_missing,BPDIAS_missing,STROKCOG_missing,CVDCOG_missing,STKIMAG_missing,CVDIMAG_missing
0,NACC000011,2006-04-17,62,2,1,0,16.0,0,0,0,...,0,0,1,0,0,0,1,1,1,1
1,NACC000034,2015-07-16,79,2,1,0,15.0,0,1,0,...,0,1,0,0,0,0,1,1,0,1
2,NACC000067,2012-05-23,60,1,1,0,18.0,0,0,0,...,0,0,1,0,0,0,1,0,1,1
3,NACC000073,2016-05-20,44,2,1,0,18.0,0,0,0,...,0,1,0,0,1,0,1,1,0,1
4,NACC000144,2010-08-17,79,1,1,0,18.0,0,1,1,...,0,0,1,0,0,0,1,0,1,1


In [34]:
# Optional: merge nearest CSF biomarkers to baseline
if not os.path.exists(NACC_CSF_PATH):
    print("CSF file not found; skipping.")
else:
    csf = pd.read_csv(NACC_CSF_PATH, low_memory=False)
    csf["NACCID"] = csf["NACCID"].astype(str)

    for c in ["CSFABETA","CSFPTAU","CSFTTAU"]:
        if c in csf.columns:
            csf[f"{c}_missing"] = csf[c].isna().astype(int)

    def make_csf_date(df_: pd.DataFrame) -> pd.Series:
        y = pd.to_numeric(df_.get("CSFLPYR"), errors="coerce")
        m = pd.to_numeric(df_.get("CSFLPMO"), errors="coerce").fillna(6).astype(int).clip(1,12)
        d = pd.to_numeric(df_.get("CSFLPDY"), errors="coerce").fillna(15).astype(int).clip(1,28)
        return pd.to_datetime(dict(year=y, month=m, day=d), errors="coerce")

    csf["csf_date"] = make_csf_date(csf)

    csf = csf.merge(X[["NACCID","baseline_date"]], on="NACCID", how="inner")
    csf = csf.dropna(subset=["csf_date","baseline_date"]).copy()
    csf["abs_days_from_baseline"] = (csf["csf_date"] - csf["baseline_date"]).abs().dt.days

    csf_nearest = csf.sort_values(["NACCID","abs_days_from_baseline"]).groupby("NACCID", as_index=False).first()

    merge_cols = ["NACCID","CSFABETA","CSFPTAU","CSFTTAU","abs_days_from_baseline"]
    merge_cols = [c for c in merge_cols if c in csf_nearest.columns]
    X = X.merge(csf_nearest[merge_cols], on="NACCID", how="left")

    print("Merged CSF. Rows:", len(X))
    if "CSFABETA" in X.columns:
        print("CSFABETA non-null:", X["CSFABETA"].notna().sum())

X.head()


Merged CSF. Rows: 37007
CSFABETA non-null: 1523


,NACCID,baseline_date,NACCAGE,SEX,RACE,HISPANIC,EDUC,DIABETES,HYPERTEN,HYPERCHO,...,BPSYS_missing,BPDIAS_missing,STROKCOG_missing,CVDCOG_missing,STKIMAG_missing,CVDIMAG_missing,CSFABETA,CSFPTAU,CSFTTAU,abs_days_from_baseline
0,NACC000011,2006-04-17,62,2,1,0,16.0,0,0,0,...,0,0,1,1,1,1,NaN,NaN,NaN,NaN
1,NACC000034,2015-07-16,79,2,1,0,15.0,0,1,0,...,0,0,1,1,0,1,NaN,NaN,NaN,NaN
2,NACC000067,2012-05-23,60,1,1,0,18.0,0,0,0,...,0,0,1,0,1,1,NaN,NaN,NaN,NaN
3,NACC000073,2016-05-20,44,2,1,0,18.0,0,0,0,...,1,0,1,1,0,1,NaN,NaN,NaN,NaN
4,NACC000144,2010-08-17,79,1,1,0,18.0,0,1,1,...,0,0,1,0,1,1,NaN,NaN,NaN,NaN


## Save cohort.csv

In [35]:
out_path = os.path.join(OUT_DIR, "cohort.csv")
X.to_csv(out_path, index=False)
print("Saved:", out_path)
print(X["label"].value_counts(dropna=False))


Saved: ..\output\cohort.csv
label
Censored         22339
NoDementia_5y    10816
AD                2948
MixedOrOther       445
PD_LBD             283
Vascular           176
Name: count, dtype: int64
